In [ ]:
import nbconvert
!jupyter nbconvert --to script _utils_siamese.ipynb

from _utils_siamese import *

In [ ]:
Class_num = 2
Size = (64, 64, 64) 
Batchsize = 8
dropout_rate = 0.5
weight_decay = 1e-4
Epoch_num = 50

adam = Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, decay=0.0)

# Parameters
params = {'dim': Size,
          'batch_size': Batchsize,
          'n_classes': Class_num,
          'n_channels': 1,
          'shuffle': True}

params_val = {'dim': Size,
              'batch_size': 1,
              'n_classes': Class_num,
              'n_channels': 1,
              'shuffle': False}

In [ ]:
# Set mixed precision training
mixed_precision.set_global_policy('mixed_float16')

model_file_current = ".../STRANet/Outcome_5fold/"

fold_num = 4   # Manually change the fold number for each training run

history_metrics = []

print(f"Training fold {fold_num}...")

fold_dir = model_file_current + f'fold_{fold_num}/'

# Load training and validation data for each fold
train_split, val_split = load_train_val_split(fold_dir)
test_split = load_test_split(".../STRANet/Data/test_data.csv")

training_generator = DataGenerator(train_split[['Pre_path', 'Post_pseudo_path', 'Binary_Label']], **params) 
validation_generator = DataGenerator(val_split[['Pre_path', 'Post_pseudo_path', 'Binary_Label']], **params_val)  
test_generator = DataGenerator(test_split[['Pre_path', 'Post_pseudo_path', 'Binary_Label']], **params_val)  

# Model initialization
model = build_siamese_CNN(Size, weight_decay, dropout_rate, Class_num)
# model = build_siamese_densenet(Size, weight_decay, dropout_rate, Class_num)
# model = build_siamese_resnet(Size, weight_decay,  dropout_rate, Class_num)

# Compile the model
model.compile(
    loss='categorical_crossentropy',
    optimizer=adam,
    metrics=['accuracy', 
             tf.keras.metrics.Precision(name='precision', class_id=1), 
             tf.keras.metrics.Recall(name='recall', class_id=1), 
             tf.keras.metrics.AUC(name='auc')])

# Define callbacks
log_file = fold_dir + f'fold{fold_num}_training.log'
csv_logger = tf.keras.callbacks.CSVLogger(log_file, append=True)

modelpath = fold_dir + 'Epoch.{epoch:02d}-Acc_{val_accuracy:.4f}-Prec_{val_precision:.4f}-Rec_{val_recall:.4f}-AUC_{val_auc:.4f}.h5'
checkpoint_auc = tf.keras.callbacks.ModelCheckpoint(modelpath, monitor='val_auc', verbose=1, save_best_only=True, mode='max')

final_model_path = fold_dir + f'final_epoch_fold{fold_num}_model.h5'
checkpoint_final_epoch = tf.keras.callbacks.ModelCheckpoint(
    filepath = final_model_path, save_best_only=False, save_weights_only=False, save_freq='epoch', verbose=1)

callbacks = [checkpoint_auc, csv_logger, checkpoint_final_epoch]

# Model training
model.fit(
    training_generator,
    validation_data=validation_generator,
    epochs=Epoch_num,
    callbacks=callbacks,
    verbose=1)

# Save predictions for training and validation sets for each fold
train_predictions, train_labels = get_predictions_and_labels(training_generator, model)
val_predictions, val_labels = get_predictions_and_labels(validation_generator, model)
test_predictions, test_labels = get_predictions_and_labels(test_generator, model)

save_predictions(fold_dir, 'train_model_predictions.csv', train_predictions, train_labels)
save_predictions(fold_dir, 'val_model_predictions.csv', val_predictions, val_labels)
save_predictions(fold_dir, 'test_model_predictions.csv', test_predictions, test_labels)

# Calculate performance metrics for the training set
train_metrics = calculate_metrics(train_predictions, train_labels)
val_metrics = calculate_metrics(val_predictions, val_labels)
test_metrics = calculate_metrics(test_predictions, test_labels)
save_metrics(fold_num, train_metrics, val_metrics, test_metrics, history_metrics)

# Clear memory
K.clear_session()  
del model  
gc.collect()  

# Save performance metrics for all folds
metrics_df = pd.DataFrame(history_metrics)
metrics_df.to_csv(model_file_current + f'fold_{fold_num}_performance_metrics.csv', index=False)